# 05. Audit the Authors' GitHub Code Before You Use It

**Author:** Md. Mobarak Karim, Ph.D.  
**Level:** Beginner → research-practical  
**Course:** Deep Learning for Optical Imaging

This notebook teaches a systematic repository audit so public code does not become an unexamined dependency in your research.

> **How to study this notebook:** read the explanation first, predict what the code should do, run it, change one parameter, and explain why the result changed.


## Learning objectives

- Audit license and environment
- Inspect data loader/model/loss/metrics
- Compare paper against code
- Test metric behavior
- Evaluate inference/postprocessing
- Make a documented reuse decision


## Mind map

```mermaid
mindmap
  root((Repository audit))
    Legal
      Code license
      Weights
      Data
    Environment
      Python
      PyTorch
      CUDA
    Data loader
      Normalize
      Resize
      Labels
      Augment
    Model
      Architecture
      Shapes
      Outputs
    Training
      Loss
      Optimizer
      Checkpoint
    Evaluation
      Metric
      Postprocess
      Split
    Decision
      Reuse
      Adapt
      Reimplement

```


## 1. Treat the repository as evidence, not authority

The released repository may be:
- the exact experiment code;
- a cleaned demonstration;
- inference-only code;
- a later refactor;
- incomplete;
- dependent on private preprocessing.

Your job is to determine what it actually contains.


## 2. Repository anatomy: where to look first

```text
README.md          claims, setup, examples
LICENSE            what you are allowed to do
requirements.*     dependencies
environment.yml    environment
configs/           hidden hyperparameters often live here
datasets.py        preprocessing and label construction
models.py          architecture
losses.py          actual objective
metrics.py         actual evaluation definition
train.py           optimizer/scheduler/checkpoint logic
infer.py           deployment preprocessing/postprocessing
weights/           checkpoints and naming
scripts/           exact commands authors ran
```

A README alone is never enough for a serious reproduction.


## 3. License audit

Before copying code, weights, or assets, check the license.

Ask separately:
- code license?
- pretrained weight license?
- dataset license?
- figures/assets license?

"Public on GitHub" does **not** automatically mean unrestricted reuse.

Record the exact license and any attribution/noncommercial/share-alike requirements relevant to your use.


## 4. Environment audit

Look for:
- Python version;
- PyTorch version;
- CUDA assumptions;
- deprecated libraries;
- compiled extensions;
- exact package pins;
- OS-specific paths;
- Docker/conda support.

### Decision
If dependencies are old, first try reproducing in an isolated environment that matches the authors. Do **not** immediately modernize everything; otherwise you cannot tell whether failure comes from the method or your changes.


## 5. Data-loader audit — usually the most important file

Search for:
- resize/crop;
- dtype conversion;
- scaling/normalization;
- channel order;
- augmentation;
- target encoding;
- patch selection;
- blank-patch rejection;
- file naming assumptions;
- train/val/test discovery.

### Example red flag
Paper: "images normalized to [0,1]"  
Code: clips 1st/99th percentiles, then scales each image separately.

Those are not the same operation.


## 6. Model audit

Compare code against the paper:
- number of blocks;
- channels;
- normalization;
- dropout;
- padding;
- activation;
- upsampling method;
- output channels;
- output activation.

Print a model summary or inspect shapes using a dummy tensor before training.


In [ ]:
import torch
import torch.nn as nn

# Example of shape probing
model = nn.Sequential(
    nn.Conv2d(1, 8, 3, padding=1),
    nn.ReLU(),
    nn.Conv2d(8, 1, 3, padding=1),
)

x = torch.randn(2, 1, 128, 128)
with torch.no_grad():
    y = model(x)

print("input :", x.shape)
print("output:", y.shape)


## 7. Loss audit

Check:
- exact formula;
- reduction (`mean`, `sum`, per-pixel);
- class weights;
- smoothing constants;
- ignored labels;
- loss coefficients;
- logits vs probabilities.

A common bug is applying sigmoid in the model **and** using `BCEWithLogitsLoss`, which expects raw logits.


## 8. Metric audit

Write a tiny known-case unit test.

Example for binary Dice:

```text
prediction = target exactly  -> Dice should be 1
no overlap                   -> Dice should approach 0
```


In [ ]:
import torch

def dice_score(pred, target, eps=1e-8):
    pred = pred.float()
    target = target.float()
    intersection = (pred * target).sum()
    return (2 * intersection + eps) / (pred.sum() + target.sum() + eps)

target = torch.tensor([1, 1, 0, 0])
print("perfect:", dice_score(target, target).item())
print("none   :", dice_score(torch.tensor([0,0,1,1]), target).item())


## 9. Training-script audit

Find:
- optimizer;
- LR;
- scheduler;
- gradient clipping;
- mixed precision;
- accumulation;
- warmup;
- checkpoint save rule;
- early stopping;
- random seed;
- validation frequency.

The "best model" may mean best validation loss, best Dice, last epoch, or manually selected checkpoint. That changes reported performance.


## 10. Inference audit

Deployment code can differ from training.

Check:
- tiling/overlap;
- padding;
- resize back to original size;
- test-time augmentation;
- threshold;
- connected-component filtering;
- color conversion;
- rescaling;
- ensemble averaging.

Postprocessing can account for a meaningful fraction of the final metric.


## 11. Reuse scorecard

Score each 0–2:

| Category | 0 | 1 | 2 |
|---|---|---|---|
| Task match | poor | partial | close |
| Data match | poor | moderate | close |
| License | unusable/unclear | restrictions | clear/compatible |
| Environment | broken | recoverable | reproducible |
| Preprocessing | hidden | partial | explicit |
| Paper/code match | poor | mixed | strong |
| Weights | none/unusable | partial | usable |
| Metrics | unclear | partial | verified |
| Split validity | concerning | uncertain | strong |
| Modularity | tangled | moderate | clean |

The score does **not** mechanically make the decision. It forces you to document the evidence.


## End-of-notebook checklist

Before moving on, you should be able to explain the main ideas **without looking at the code**. If you cannot explain why a method, loss, split, or metric is appropriate, repeat the relevant section before using it in research.
